In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Preparing data
df = pd.read_csv("All_MDD_LLM.csv")

# choose which dataset to use
# feature_cols = df.columns[1:25].tolist() + [df.columns[37]] # Financial
feature_cols = df.columns[1:38].tolist() # Financial + GPT
# feature_cols = df.columns[1:38].tolist() + df.columns[47:].tolist() # Financial + GPT + LLM
target_cols = df.columns[38:47]

X = df[feature_cols].values.astype(np.float32)
instances = df["instance"].values

all_actual = pd.DataFrame(index=df.index, columns=target_cols)
all_pred = pd.DataFrame(index=df.index, columns=target_cols)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Define LSTM Model
class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, output_dim=1):
        super(LSTMRegressor, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # take the last time step
        out = self.fc(out)
        return out

# KFold Loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for target in target_cols:
    y = df[target].values.astype(np.float32)

    for train_idx, test_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[test_idx]
        y_train, y_val = y[train_idx], y[test_idx]

        X_train_t = torch.tensor(X_train).unsqueeze(1).to(device)
        y_train_t = torch.tensor(y_train).unsqueeze(1).to(device)
        X_val_t = torch.tensor(X_val).unsqueeze(1).to(device)
        y_val_t = torch.tensor(y_val).unsqueeze(1).to(device)

        train_ds = TensorDataset(X_train_t, y_train_t)
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

        # init the model
        model = LSTMRegressor(input_dim=X.shape[1]).to(device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

        # Train with Early Stopping
        best_loss = float("inf")
        patience, wait = 5, 0  # stop if no improvement in 5 rows
        best_state = None

        for epoch in range(100):  # 100 epoch maximum
            model.train()
            for xb, yb in train_loader:
                optimizer.zero_grad()
                pred = model(xb)
                loss = criterion(pred, yb)
                loss.backward()
                optimizer.step()

            # evaluate the model
            model.eval()
            with torch.no_grad():
                val_pred = model(X_val_t)
                val_loss = criterion(val_pred, y_val_t).item()

            if val_loss < best_loss:
                best_loss = val_loss
                wait = 0
                best_state = model.state_dict()
            else:
                wait += 1
                if wait >= patience:
                    # print(f"Early stopping at epoch {epoch}, best val_loss={best_loss:.4f}")
                    break

        # reflect to the best model
        model.load_state_dict(best_state)

        # test
        model.eval()
        with torch.no_grad():
            y_pred = model(X_val_t).cpu().numpy().flatten()

        # result
        all_actual.loc[test_idx, target] = y_val
        all_pred.loc[test_idx, target] = y_pred

# Statistic
result_summary = pd.DataFrame(index=df.index)
result_summary["Instance"] = instances
result_summary["SBS"] = all_actual["OH-L"]
result_summary["VBS"] = all_actual.max(axis=1)

best_pred_algo = all_pred.idxmax(axis=1)
result_summary["Selected Algo"] = best_pred_algo
selection_result = []
for i, algo in best_pred_algo.items():
    selection_result.append(all_actual.loc[i, algo])
result_summary["Selection Result"] = selection_result

stats = pd.DataFrame({
    "Mean": result_summary[["SBS", "VBS", "Selection Result"]].mean(),
    "Std": result_summary[["SBS", "VBS", "Selection Result"]].std(),
    "Min": result_summary[["SBS", "VBS", "Selection Result"]].min(),
    "Max": result_summary[["SBS", "VBS", "Selection Result"]].max(),
}).T

algo_counts = result_summary["Selected Algo"].value_counts().rename_axis("Algorithm").reset_index(name="Count")
algo_counts["Proportion"] = algo_counts["Count"] / len(result_summary)

# save
result_summary.to_csv("MDD_strategy_selection_LSTM.csv", index=False)
stats.to_csv("MDD_strategy_selection_stats_LSTM.csv")
algo_counts.to_csv("MDD_strategy_selection_algo_counts_LSTM.csv", index=False)

print("Done with Early Stopping!")


Done with Early Stopping!
